# 🚀 QLoRA Fine-Tuning on Free Tier (Colab / Kaggle)

This notebook runs the **entire pipeline** on a free T4 GPU.

| Platform | GPU | VRAM | Time limit | Setup |
|----------|-----|------|------------|-------|
| **Kaggle** | 2x T4 | 16 GB | 30 hrs/week | Accelerator -> GPU T4 x2 |
| **Colab Free** | T4 | 15 GB | ~4 hrs session | Runtime -> T4 GPU |

**Estimated run time**: ~90 min (20K samples, 1 epoch on T4)

---
## 0 - Setup and Install

In [ ]:
# Clone repo
!git clone https://github.com/visheshgupta29/llm-lora-finetuning.git
%cd llm-lora-finetuning

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# Verify GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'bfloat16 support: {torch.cuda.is_bf16_supported()}')

In [ ]:
# Set secrets
import os

# Try Kaggle secrets first, then Colab, then env
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
    os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
    print('Loaded from Kaggle secrets')
except Exception:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
        os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
        print('Loaded from Colab secrets')
    except Exception:
        print('Set HF_TOKEN and WANDB_API_KEY manually')

# Optional: skip W&B if you don't want to set it up
# os.environ['WANDB_MODE'] = 'disabled'

---
## 1 - Prepare Dataset

In [ ]:
# Use the T4-optimized config
CONFIG = 'configs/training_config_t4.yaml'

!python -m src.data.prepare_dataset --config {CONFIG}

In [ ]:
# Quick sanity check
import json
with open('data/processed/train.jsonl') as f:
    lines = f.readlines()
print(f'Training examples: {len(lines):,}')
print(f'\nFirst example preview:')
print(json.loads(lines[0])['text'][:500])

---
## 2 - Train with QLoRA

This is the main training cell. Takes **~90 min** on T4 (20K samples, 1 epoch).

In [ ]:
# Check VRAM before training
!nvidia-smi --query-gpu=name,memory.total,memory.used,memory.free --format=csv

In [ ]:
# Launch training
!python -m src.train.finetune_lora --config {CONFIG}

In [ ]:
# VRAM after training
import torch
print(f'Peak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB')
print(f'Current VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

---
## 3 - Evaluate

In [ ]:
# Free VRAM from training before loading for eval
import gc
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM after cleanup: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

In [ ]:
!python -m src.evaluate.evaluate_model \
    --adapter-path outputs/final-adapter \
    --test-split data/processed/test.jsonl \
    --num-samples 100 \
    --run-execution-accuracy \
    --output-file outputs/eval_results.json

In [ ]:
# View results
import json
from collections import Counter

with open('outputs/eval_results.json') as f:
    payload = json.load(f)

summary = payload.get('summary', {})
results = payload.get('results', [])

print('Evaluation Summary')
print('=' * 50)
print(f"Samples                : {summary.get('num_samples', len(results))}")
print(f"Exact match rate       : {summary.get('exact_match_rate', 0):.2f}%")
print(f"Valid SQL rate         : {summary.get('valid_sql_rate', 0):.2f}%")
print(f"Execution accuracy     : {summary.get('execution_accuracy_rate', 0):.2f}%")
print(f"Average BLEU           : {summary.get('avg_bleu', 0):.4f}")

error_dist = summary.get('error_distribution')
if not error_dist and results:
    error_dist = dict(Counter(r.get('error_type', 'unknown') for r in results))

if error_dist:
    print('\nError Distribution')
    print('-' * 50)
    for err, count in sorted(error_dist.items(), key=lambda x: x[1], reverse=True):
        print(f"{err:20s} : {count}")

print('\nSample Predictions (first 3)')
print('-' * 50)
for r in results[:3]:
    print(f"Q: {r.get('question', '')}")
    print(f"Gold: {r.get('gold_sql', '')}")
    print(f"Pred: {r.get('predicted_sql', '')[:220]}...")
    print(f"exec_ok={r.get('execution_accuracy', False)}, bleu={r.get('bleu', 0):.4f}, error={r.get('error_type', 'n/a')}")
    print()

---
## 4 - Interactive Predictions

In [ ]:
from src.inference.predict import SQLPredictor

predictor = SQLPredictor(
    adapter_path='outputs/final-adapter',
    base_model_name='mistralai/Mistral-7B-v0.3',
)

In [ ]:
# Try it out!
schema = '''CREATE TABLE employees (
    id INT PRIMARY KEY,
    name TEXT,
    department TEXT,
    salary REAL,
    hire_date TEXT
);'''

questions = [
    'How many employees are in the Engineering department?',
    'What is the average salary by department?',
    'List the top 5 highest paid employees.',
    'Who was hired most recently?',
]

for q in questions:
    sql = predictor.predict(question=q, schema=schema)
    print(f'Q: {q}')
    print(f'SQL: {sql}')
    print()

---
## 5 - Save Adapter to Hugging Face Hub

Push the ~30 MB adapter so you can load it anywhere later.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.upload_folder(
    folder_path='outputs/final-adapter',
    repo_id='visheshgupta29/mistral-7b-text2sql-qlora',
    repo_type='model',
    commit_message='Upload QLoRA adapter trained on sql-create-context',
)
print('Adapter uploaded to HF Hub!')

---
## 6 - Launch Gradio Demo (Colab only)

This creates a **public URL** you can share.

In [ ]:
from src.inference.serve import build_demo

demo = build_demo(predictor)
demo.launch(share=True)  # Creates a public gradio.live link

---
### Results so far

| Run | Exec Acc | BLEU | Notes |
|-----|----------|------|-------|
| Base Mistral-7B-v0.3 | 28.5% | 0.199 | No fine-tuning |
| v1 (1 epoch, `do_sample=True`, rep_penalty=1.1) | 8.0% | 0.082 | Repetition loops dominate |
| v2 (greedy, rep_penalty=1.3, early stopped ep 0.96) | 22.0% | 0.096 | Still loops — correct SQL is in output but truncated |

**Key insight:** v2 generates the exact correct SQL first, then loops. Fixing the stopping condition should push it well above the 28.5% base.

### Next Steps — v3 (no retraining needed)

```python
# Option A: post-processing fast win (try this first — zero cost)
# Extract first complete statement before the loops start
sql = prediction.split(";")[0].strip() + ";"

# Option B: inference config changes in training_config_t4.yaml
generation:
  no_repeat_ngram_size: 4   # hard-blocks any 4-token sequence from repeating
  repetition_penalty: 1.5   # up from 1.3
  max_new_tokens: 128       # already set
```

1. **Post-process predictions** — `split(";")[0]` before execution; re-run `evaluate_model` on existing adapter
2. **Add `no_repeat_ngram_size: 4`** to `training_config_t4.yaml` generation config
3. **Add prompt instruction** — prepend `-- Write a single concise SQL query.` to the schema block
4. **Push adapter to HF Hub** — `api.upload_folder(folder_path="outputs/final-adapter", ...)`
5. **Commit results** — `git add . && git commit -m "docs: add v2 eval + base comparison results"`
